<a href="https://colab.research.google.com/github/Eswa2020/urban-parking-prediction-models/blob/master/notebooks/Melbourne/02_melbourne_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing libraries and loading dataset

In [ ]:
#importimg our libraries to use
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from datetime import datetime
%matplotlib inline

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
#let's load the  dataset
dataq=pd.read_csv("/content/drive/MyDrive/melbourne_parking_data.csv",encoding='ISO-8859-1')
dataq.head(2)

In [ ]:
dataq.tail(2)

The dataset contains parking sensor records from 2019 and appears to be concentrated in Docklands, Melbourne, Australia.

## Exploring our dataset

In [ ]:
#We have about 1,048 ,575 rows and 20 columns.
# The dataset contains over 1.04 million valid parking records, which is large enough for robust temporal and operational analysis.
dataq.shape

In [ ]:
#We check if we have any null values in our columns
#sign plate has like 20% which is expected behaviour
dataq.isna().sum()

In [ ]:
#W have like 407 duplicates...should recheck thsi
dataq.duplicated().sum()

In [ ]:
#We have several interger columns none seems to be out of place
#we have 2 categorical(and also potential target variables "vehicle present and Inviolation")
dataq.dtypes


In [ ]:
#WE have 8 columns that are intergers
descriptive_stats = dataq.describe()
print(descriptive_stats)

In [ ]:
# DurationMinutes has a mean of about 48 minutes, indicating that the average parking event lasts less than one hour.
# The median duration is 12 minutes, which is much lower than the mean and suggests a strongly right-skewed distribution.
# The standard deviation of about 92.6 minutes shows substantial variation in parking duration across events.
# The minimum duration is 0 minutes, which may reflect very short stays, sensor noise, or recording artefacts.
# The 25th percentile is 3 minutes, meaning that a quarter of all parking events are extremely short.
# The 75th percentile is 46 minutes, meaning that most parking events are under one hour in duration.
# The maximum duration is 1,440 minutes, indicating the presence of very long stays or potential outliers in the dataset.

In [ ]:
# VehiclePresent frequency shows how often bays are recorded as occupied versus unoccupied in the dataset.
# InViolation frequency shows how often parking events are recorded as violating parking rules.
# These distributions help assess class balance before any classification or operational analysis.

vehicle_status_freq = dataq['VehiclePresent'].value_counts()
in_violation_freq = dataq['InViolation'].value_counts()

print("Vehicle Presence Frequency:")
print(vehicle_status_freq)

print("\nIn Violation Frequency:")
print(in_violation_freq)

In [ ]:
# VehiclePresent is fairly balanced, with 537,207 True values and 511,368 False values, indicating a near-even split between occupied and unoccupied observations.
# InViolation is strongly skewed toward non-violations, with 993,420 False values versus 55,155 True values, indicating that parking violations are relatively rare events in the dataset.


##Cleaning Data

In [ ]:
# Data cleaning prepares the raw parking records for reliable analysis and modeling.
# This step removes duplicates, fixes datetime fields, checks missing values, and standardizes key variables.

In [ ]:
# Create a working copy to avoid changing the original dataset.
df = dataq.copy()


In [ ]:
# Check dataset shape before cleaning.
print("Initial shape:", df.shape)

In [ ]:
# Remove duplicate records to avoid repeated parking events influencing the analysis.
df = df.drop_duplicates()

In [ ]:
# Convert datetime columns into proper datetime format for time-based analysis.
df["ArrivalTime"] = pd.to_datetime(df["ArrivalTime"], errors="coerce")
df["DepartureTime"] = pd.to_datetime(df["DepartureTime"], errors="coerce")

In [ ]:
# Check missing values in the main variables used for analysis.
missing_summary = df[[
    "ArrivalTime", "DepartureTime", "DurationMinutes",
    "AreaName", "StreetName", "BayId", "VehiclePresent", "InViolation"
]].isnull().sum()

print("\nMissing values in key columns:")
print(missing_summary)

In [ ]:
# Drop rows with missing values in essential analytical fields.
df = df.dropna(subset=[
    "ArrivalTime", "DepartureTime", "DurationMinutes",
    "AreaName", "BayId", "VehiclePresent", "InViolation"
])

In [ ]:
# Keep only non-negative durations because negative parking time is not valid.
df = df[df["DurationMinutes"] >= 0]

In [ ]:
# Convert boolean status fields into integer form for easier aggregation and modeling.
df["VehiclePresent"] = df["VehiclePresent"].astype(int)
df["InViolation"] = df["InViolation"].astype(int)

In [ ]:
# Check dataset shape after cleaning.
print("\nCleaned shape:", df.shape)

# Display data types after cleaning.
print("\nData types after cleaning:")
print(df.dtypes)


In [ ]:
# Preview the cleaned dataset.
df.head(2)

##Feature Enginnering

In [ ]:
# Exploratory data analysis is used to understand the distribution, timing, and operational patterns of parking activity.
# This step helps identify skewness, temporal trends, class balance, and area-level variation before modeling.

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create core temporal features for exploratory analysis.
df["date"] = df["ArrivalTime"].dt.date
df["month"] = df["ArrivalTime"].dt.month
df["day_of_week"] = df["ArrivalTime"].dt.day_name()
df["arrival_hour"] = df["ArrivalTime"].dt.hour


In [ ]:
# Create a weekday/weekend variable for clearer operational comparison.
df["day_type"] = np.where(df["ArrivalTime"].dt.dayofweek >= 5, "Weekend", "Weekday")

In [ ]:
# Create a season variable using Australian seasonal grouping.
def get_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    else:
        return "Spring"

df["season"] = df["month"].apply(get_season)

#Forecasting Layer

**Forecasting Objective**

This notebook examines whether short-term parking occupancy in Docklands can be forecast more effectively using temporal structure than by relying on a simple persistence assumption. Because the dataset is overwhelmingly concentrated in Docklands, the forecasting workflow is defined at the street-hour level rather than as a citywide Melbourne model.

The forecasting target is the street-hour occupancy rate, constructed by aggregating parking events by street and hourly timestamp. This unit of analysis is appropriate because it captures short-run operational variation while reducing the noise of raw event-level observations.

In [ ]:
# Forecasting is used to estimate short-term parking dynamics over time using aggregated street-hour operational patterns.

In [ ]:
#restricted forecasting analysis to the dominant study area to avoid weak signals from very small peripheral groups.
docklands_df = df[df["AreaName"].str.contains("Docklands", case=False, na=False)].copy()

In [ ]:
# Created a full datetime hour variable for time-based aggregation.
docklands_df["arrival_hour_ts"] = docklands_df["ArrivalTime"].dt.floor("H")

In [ ]:
# Aggregated parking activity by street and hour.
street_hour = (
    docklands_df.groupby(["StreetName", "arrival_hour_ts"], as_index=False)
    .agg(
        occupancy_rate=("VehiclePresent", "mean"),
        mean_duration=("DurationMinutes", "mean"),
        event_count=("BayId", "count"),
        violation_rate=("InViolation", "mean")
    )
    .sort_values(["StreetName", "arrival_hour_ts"])
)

print(street_hour.shape)
street_hour.head()

In [ ]:
#created Lag features

In [ ]:
# Lag features capture short-run temporal dependence in parking activity.
street_hour = street_hour.sort_values(["StreetName", "arrival_hour_ts"]).copy()

street_hour["lag_1_occ"] = street_hour.groupby("StreetName")["occupancy_rate"].shift(1)
street_hour["lag_2_occ"] = street_hour.groupby("StreetName")["occupancy_rate"].shift(2)
street_hour["lag_24_occ"] = street_hour.groupby("StreetName")["occupancy_rate"].shift(24)

street_hour["hour"] = street_hour["arrival_hour_ts"].dt.hour
street_hour["day_of_week"] = street_hour["arrival_hour_ts"].dt.dayofweek
street_hour["month"] = street_hour["arrival_hour_ts"].dt.month

street_hour = street_hour.dropna().copy()

print(street_hour.shape)
street_hour.head()

#Forecasting Model Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Select features for forecasting occupancy rate.
X = street_hour[["lag_1_occ", "lag_2_occ", "lag_24_occ", "hour", "day_of_week", "month"]]
y = street_hour["occupancy_rate"]

In [ ]:
# Use a time-ordered split instead of random shuffling.
split_index = int(len(street_hour) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
# Naive baseline using previous-hour occupancy.
baseline_pred = X_test["lag_1_occ"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = mean_squared_error(y_test, baseline_pred) ** 0.5

**Forecasting Model Design**

Two forecasting approaches are compared.
 * The first is a naive baseline that assumes the next observed occupancy rate is equal to the previous hour’s value. This provides a simple persistence benchmark.
 * The second is a Random Forest regression model that uses lagged occupancy and temporal features to capture more complex and potentially nonlinear forecasting structure.

A time-ordered train-test split is used instead of random shuffling so that the evaluation better reflects real forecasting conditions and avoids leakage from future observations into the training data.

In [ ]:
# Random Forest forecasting model.
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = mean_squared_error(y_test, rf_pred) ** 0.5

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("RF MAE:", rf_mae)
print("RF RMSE:", rf_rmse)

In [ ]:
## The Random Forest forecasting model substantially improves on the naive baseline, indicating that lagged occupancy and temporal features contain useful predictive signal for Docklands street-hour parking dynamics.

**Forecasting Results Interpretation**

The forecasting results show that the Random Forest model materially outperforms the naive lag baseline. The baseline produced an MAE of 0.2823 and an RMSE of 0.4011, whereas the Random Forest reduced these to 0.1736 and 0.2466 respectively.

*This indicates that Docklands parking occupancy contains useful short-run structure that is not fully captured by a simple persistence rule.*

In practical terms, the model benefits from combining lagged occupancy with broader temporal features such as hour of day, day of week, and month. This suggests that street-hour parking conditions in Docklands are patterned rather than purely random and can therefore support operational forecasting.

In [ ]:
import matplotlib.pyplot as plt

# Compare actual and predicted occupancy rates for a short test window.
plot_df = pd.DataFrame({
    "actual": y_test.values[:200],
    "baseline": baseline_pred.values[:200],
    "rf_pred": rf_pred[:200]
})

plt.figure(figsize=(12, 5))
plt.plot(plot_df["actual"], label="Actual")
plt.plot(plot_df["baseline"], label="Baseline")
plt.plot(plot_df["rf_pred"], label="Random Forest")
plt.title("Forecasting Docklands Street-Hour Occupancy Rate")
plt.xlabel("Test Observations")
plt.ylabel("Occupancy Rate")
plt.legend()
plt.show()

In [ ]:
#the baseline tracks the sharp spikes and drops too literally because it just copies the last observed hour, while the Random Forest is smoother and closer to the central occupancy pattern. So the baseline overreacts to volatility, whereas the RF generalizes better.

**Forecast Plot Interpretation**

The comparison plot shows that the naive baseline tends to track sharp swings too literally because it simply carries the previous-hour signal forward.

 By contrast, the Random Forest model produces smoother predictions that align more closely with the central occupancy pattern across the test period.

 This suggests that the feature-based model generalizes better to short-term parking variation.

#Conclusion

This forecasting notebook shows that short-term Docklands street-hour occupancy can be predicted more effectively with a feature-based machine-learning model than with a naive persistence benchmark.

The improvement in both MAE and RMSE indicates that lagged occupancy and temporal variables contain meaningful predictive signal.

These findings support the use of structured street-hour forecasting as a component of urban parking decision support. However, the results should be interpreted as Docklands-specific rather than representative of all Melbourne, given the strong spatial concentration of the dataset.